In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
T-Forecast V8.0 - 即時新聞爬蟲與自動標註系統 (完整修復版)
修正重點:
1. [資料安全] 確保 Publish Time 優先抓取並儲存，內文清洗不影響時間欄位。
2. [萬用解析] 針對 LTN/SETN 子網域 (體育/娛樂/財經) 實作多重容器掃描。
3. [自動標註] 讀取舊資料 (Label 1)，新抓取的即時新聞標記為 Label 0。
"""

import os
import sys
import time
import random
import logging
import pandas as pd
import re
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# ===========================
# 1. 專案設定 (Config)
# ===========================

DATA_FILENAME = 'news_dataset_v5.csv'
LOG_FILENAME = 'scraper_v8.log'

# 即時新聞列表網址
REALTIME_URLS = {
    "TVBS": "https://news.tvbs.com.tw/realtime",
    "UDN": "https://udn.com/news/breaknews/1",
    "ChinaTimes": "https://www.chinatimes.com/realtimenews/?chdtv",
    "LTN": "https://news.ltn.com.tw/list/breakingnews",
    "SETN": "https://www.setn.com/ViewAll.aspx?PageGroupID=0",
    "ETtoday": "https://www.ettoday.net/news/news-list.htm"
}

# 捲動次數：設為 30 次以抓取大量 Label 0
SCROLL_TIMES = 30 
RESTART_FREQUENCY = 50

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler(LOG_FILENAME, encoding='utf-8'), logging.StreamHandler(sys.stdout)]
)

# ===========================
# 2. 基礎工具
# ===========================

def format_output_time(dt_obj=None):
    if dt_obj is None: dt_obj = datetime.now()
    return dt_obj.strftime("%Y/%m/%d %I:%M:%S %p").lower()

def clean_text(text):
    if not text: return "N/A"
    text = re.sub(r'\s+', ' ', text).strip()
    garbage = ["請繼續往下閱讀...", "點我下載APP", "按我看活動辦法", "免費訂閱", "相關新聞", "看更多"]
    for g in garbage:
        text = text.replace(g, "")
    return text

def ensure_newline_in_csv(filename):
    if not os.path.exists(filename): return
    try:
        with open(filename, 'rb+') as f:
            f.seek(0, 2)
            if f.tell() > 0:
                f.seek(-1, 2)
                if f.read(1) != b'\n':
                    f.write(b'\n')
    except: pass

def parse_time_string(time_str):
    if not time_str or time_str == "N/A": return "N/A"
    clean = time_str.strip()
    # ISO
    if 'T' in clean:
        try:
            dt_part = clean.split('T')[0] + ' ' + clean.split('T')[1].split('+')[0].split('.')[0].replace('Z', '')
            return format_output_time(datetime.strptime(dt_part, "%Y-%m-%d %H:%M:%S"))
        except: pass
    # 一般格式
    clean_nums = re.sub(r'[^\d\-\:\/\s]', '', clean).strip()
    formats = ["%Y/%m/%d %H:%M:%S", "%Y-%m-%d %H:%M:%S", "%Y/%m/%d %H:%M", "%Y-%m-%d %H:%M"]
    for fmt in formats:
        try: return format_output_time(datetime.strptime(clean_nums[:19], fmt))
        except: continue
    # 無年份
    try:
        dt = datetime.strptime(clean_nums, "%m/%d %H:%M")
        dt = dt.replace(year=datetime.now().year)
        return format_output_time(dt)
    except: pass
    return "N/A"

def get_driver():
    opts = Options()
    opts.add_argument("--headless")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    opts.add_experimental_option("prefs", {"profile.managed_default_content_settings.images": 2})
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

def human_scroll_deep(driver, times=10):
    logging.info(f"   🔄 正在深度捲動 {times} 次以載入更多新聞...")
    last_height = driver.execute_script("return document.body.scrollHeight")
    for i in range(times):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(random.uniform(2.0, 3.5))
        if i % 5 == 0: driver.execute_script("window.scrollBy(0, -300)")
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            time.sleep(2)
            if driver.execute_script("return document.body.scrollHeight") == new_height: break
        last_height = new_height

# ===========================
# 3. 萬用解析核心 (Universal Parser)
# ===========================

def extract_meta(soup, prop):
    tag = soup.find('meta', property=prop) or soup.find('meta', attrs={'name': prop}) or soup.find('meta', attrs={'itemprop': prop})
    return tag['content'].strip() if tag and tag.get('content') else None

def get_selector_text(soup, selectors):
    for sel in selectors:
        tag = soup.select_one(sel)
        if tag: return clean_text(tag.get_text())
    return None

# --- [關鍵] LTN 萬用解析器 ---
def parse_ltn_universal(soup, url):
    # 1. 標題
    title = extract_meta(soup, 'og:title')
    if not title:
        title = get_selector_text(soup, ['.news_content h1', 'div.whitecon h1', '.article-title', 'h1'])
    
    # 2. 類別
    category = extract_meta(soup, 'article:section')
    if not category: 
        category = get_selector_text(soup, ['.breadcrumbs a:nth-child(2)', '.guide a:nth-child(2)', '.boxTitle .label'])

    # 3. 時間 (優先抓取，確保不被清洗掉)
    pub_time = extract_meta(soup, 'article:published_time')
    if not pub_time:
        pub_time = get_selector_text(soup, ['.c_time', '.mobile_time', 'span.time', '.time'])

    # 4. 內文 (多重容器掃描)
    content = "N/A"
    potential_containers = [
        '.news_content', '.whitecon .text', 'div.text', 
        'div[itemprop="articleBody"]', '.article-content', 'main'
    ]
    
    target_element = None
    for selector in potential_containers:
        target_element = soup.select_one(selector)
        if target_element and len(target_element.get_text(strip=True)) > 30:
            break
            
    if target_element:
        # 移除雜訊 (這裡移除 .c_time 不會影響上面的 pub_time，因為 pub_time 已經存下來了)
        for trash in target_element.select('.c_time, .author, .h_promo, .app_ad, .suggest_box, script, style, .onead, .pic_video'):
            trash.decompose()
            
        paragraphs = target_element.find_all('p')
        valid_paragraphs = [p.get_text(strip=True) for p in paragraphs if len(p.get_text(strip=True)) > 1]
        
        if valid_paragraphs:
            content = "\n".join(valid_paragraphs)
        else:
            content = target_element.get_text(separator='\n', strip=True)
            
    content = clean_text(content)

    return title, content, category, parse_time_string(pub_time)

# --- SETN 解析器 ---
def parse_setn_universal(soup, url):
    title = extract_meta(soup, 'og:title')
    if not title: title = get_selector_text(soup, ['h1.news-title-3', 'h1.page-title-text', 'h1'])
    
    category = extract_meta(soup, 'article:section')
    
    pub_time = extract_meta(soup, 'article:published_time')
    if not pub_time: pub_time = get_selector_text(soup, ['time.page-date', '.news-time', '.date'])

    content = "N/A"
    containers = ['.page-text', '.news-content', 'article', 'div#Content1']
    for sel in containers:
        tag = soup.select_one(sel)
        if tag and len(tag.get_text()) > 30:
            content = clean_text(tag.get_text(separator='\n'))
            break
            
    return title, content, category, parse_time_string(pub_time)

# --- 通用解析器 ---
def parse_general(soup, media):
    title = extract_meta(soup, 'og:title')
    category = extract_meta(soup, 'article:section')
    pub_time = extract_meta(soup, 'article:published_time')
    content = "N/A"
    
    if media == "TVBS":
        content = get_selector_text(soup, ['#news_detail_div', '.article_content'])
        if not pub_time: pub_time = get_selector_text(soup, ['.author .time'])
    elif media == "UDN":
        content = get_selector_text(soup, ['section.article-content__editor'])
        if not pub_time: pub_time = get_selector_text(soup, ['.article-content__time'])
    elif media == "ChinaTimes":
        content = get_selector_text(soup, ['.article-body'])
        if not pub_time: pub_time = get_selector_text(soup, ['.meta-info time'])
    elif media == "ETtoday":
        content = get_selector_text(soup, ['div.story', '.fncnews-content'])
        if not pub_time: pub_time = get_selector_text(soup, ['time.date'])
        
    return title, content, category, parse_time_string(pub_time)

# ===========================
# 4. 主邏輯 (自動標記)
# ===========================

def get_realtime_links(driver, media):
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    links = set()
    selectors = {
        "TVBS": '.news_list .list li a',
        "UDN": '.story-list__news h2 a',
        "ChinaTimes": '.vertical-list li h3.title a',
        "LTN": ['ul.list li a.tit', 'ul.list li > a'],
        "SETN": '.NewsList h3 a',
        "ETtoday": '.part_list_2 h3 a'
    }
    sel = selectors.get(media)
    items = []
    if isinstance(sel, list):
        for s in sel: items += soup.select(s)
    else:
        items = soup.select(sel)

    for item in items:
        href = item.get('href')
        if href:
            if href.startswith('//'): href = 'https:' + href
            elif href.startswith('/'):
                domain = REALTIME_URLS[media].split('/')[2]
                href = f"https://{domain}" + href
            if re.search(r'\d+', href) and not any(x in href for x in ['search', 'category', 'live']):
                links.add(href)
    return list(links)

def run_spider():
    logging.info("======= V8.0 即時新聞爬蟲與自動標記啟動 =======")
    
    # 1. 建立熱門白名單 (Hot List)
    hot_urls_set = set()
    if os.path.exists(DATA_FILENAME):
        try:
            df_old = pd.read_csv(DATA_FILENAME)
            if 'label' not in df_old.columns:
                logging.info("🔧 偵測到舊資料無標籤，自動標記為 1 (熱門)...")
                df_old['label'] = 1
                df_old.to_csv(DATA_FILENAME, index=False, encoding='utf-8-sig')
            
            hot_urls_set = set(df_old[df_old['label'] == 1]['url'].tolist())
            logging.info(f"📚 已載入 {len(hot_urls_set)} 筆熱門新聞白名單")
        except Exception as e:
            logging.error(f"讀取舊資料失敗: {e}")

    driver = get_driver()
    new_data = []
    
    try:
        for media, url in REALTIME_URLS.items():
            logging.info(f"🚀 [{media}] 掃描即時新聞: {url}")
            try:
                driver.get(url)
                human_scroll_deep(driver, times=SCROLL_TIMES)
                
                links = get_realtime_links(driver, media)
                logging.info(f"   -> 找到 {len(links)} 篇候選連結")
                
                count = 0
                for i, link in enumerate(links):
                    if link in hot_urls_set: continue
                        
                    if count > 0 and count % RESTART_FREQUENCY == 0:
                        logging.info("   ♻️ 重啟瀏覽器...")
                        driver.quit()
                        driver = get_driver()
                    
                    logging.info(f"   ({i+1}/{len(links)}) 抓取一般新聞 (Label 0): {link}")
                    try:
                        driver.get(link)
                        time.sleep(random.uniform(1.5, 2.5))
                        soup = BeautifulSoup(driver.page_source, 'html.parser')
                        
                        if media == "LTN": data = parse_ltn_universal(soup, link)
                        elif media == "SETN": data = parse_setn_universal(soup, link)
                        else: data = parse_general(soup, media)
                            
                        title, content, category, pub_time = data
                        
                        if content and len(content) > 30 and content != "N/A":
                            if not title: title = driver.title
                            new_data.append({
                                'url': link, 'media': media, 'category': category,
                                'publish_time': pub_time, 'title': title, 'content': content,
                                'fetch_time': format_output_time(),
                                'label': 0 
                            })
                            count += 1
                        else:
                            logging.warning(f"   ⚠️ 內容無效")
                    except Exception as e:
                        logging.error(f"   🔥 錯誤: {e}")

            except Exception as e:
                logging.error(f"   🔥 列表錯誤: {e}")
    
    finally:
        driver.quit()

    if new_data:
        ensure_newline_in_csv(DATA_FILENAME)
        df_new = pd.DataFrame(new_data)
        header = not os.path.exists(DATA_FILENAME)
        df_new.to_csv(DATA_FILENAME, mode='a', header=header, index=False, encoding='utf-8-sig')
        logging.info(f"✅ 任務完成！新增 {len(new_data)} 筆一般新聞 (Label 0)")
    else:
        logging.info("⚠️ 本次無新資料寫入")

if __name__ == "__main__":
    run_spider()